In [1]:
import pandas as pd
import mysql.connector

con = mysql.connector.connect(
    user='root',
    host='localhost',
    password='',
    database='import_practice'
)


C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Problem 1-4 : NorthWinddatset https://relational.fit.cvut.cz/dataset/Northwind

How to download the dataset

To download the dataset you need to create connection in your mysql workbench

The datasets are publicly available directly from MariaDB database.

* Open your favourite MariaDB client (MySQL Workbench)
Use following credentials:
```
    hostname: relational.fit.cvut.cz
    port: 3306
    username: guest
    password: relational
```
Drive link: https://drive.google.com/drive/folders/1FMi6g1KDHbn17ikJSi7bE9-THywlxhdu?usp=sharing

### `Q-1`: Rank Employee in terms of revenue generation. Show employee id, first name, revenue, and rank


### `Q-2`: Show All products cumulative sum of units sold each month.


### `Q-3`: Show Percentage of total revenue by each suppliers



### `Q-4:` Show Percentage of total orders by each suppliers



### `Q-5`:Show All Products Year Wise report of totalQuantity sold, percentage change from last year.





In [2]:
query = """SELECT *, RANK() OVER(ORDER BY revenue) FROM
(SELECT nw_employees.FirstName,nw_employees.EmployeeID,SUM(UnitPrice*Quantity) AS 'revenue' FROM nw_orders
JOIN nw_order_details
ON nw_orders.OrderID=nw_order_details.OrderID
JOIN nw_employees
ON nw_orders.EmployeeID=nw_employees.EmployeeID
GROUP BY nw_employees.EmployeeID) AS t;"""


pd.read_sql_query(query, con=con)

C:\Users\HP\AppData\Local\Temp\ipykernel_73500\3304230362.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query(query, con=con)


,FirstName,EmployeeID,revenue,RANK() OVER(ORDER BY revenue)
0,Anne,9,18885.30,1
1,Michael,6,37812.64,2
2,Steven,5,39327.40,3
3,Laura,8,56056.00,4
4,Robert,7,59541.60,5
5,Andrew,2,66519.00,6
6,Janet,3,86548.05,7
7,Nancy,1,92335.38,8
8,Margaret,4,132910.29,9


In [3]:
query = """SELECT ProductName, MONTHNAME(nw_orders.OrderDate),	Quantity, 
SUM(Quantity) OVER(partition by ProductID ORDER BY MONTHNAME(nw_orders.OrderDate) 
rows between unbounded preceding and current row)
FROM nw_order_details
JOIN nw_products
ON nw_order_details.ProductID=nw_products.ProductID
JOIN nw_orders
ON nw_orders.OrderID=nw_order_details.OrderID
GROUP BY nw_products.ProductID, MONTHNAME(nw_orders.OrderDate);"""


pd.read_sql_query(query, con=con)

C:\Users\HP\AppData\Local\Temp\ipykernel_73500\4183847610.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query(query, con=con)


,ProductName,MONTHNAME(nw_orders.OrderDate),Quantity,SUM(Quantity) OVER(partition by ProductID ORDER BY MONTHNAME(nw_orders.OrderDate) \nrows between unbounded preceding and current row)
0,Chai,April,40,40.0
1,Chai,August,45,85.0
2,Chai,December,15,100.0
3,Chai,January,10,110.0
4,Chai,July,20,130.0
...,...,...,...,...
580,Original Frankfurter grne Soe,July,12,131.0
581,Original Frankfurter grne Soe,June,18,149.0
582,Original Frankfurter grne Soe,March,30,179.0
583,Original Frankfurter grne Soe,May,10,189.0


In [5]:
query = """WITH t1 AS (SELECT nw_suppliers.SupplierID, SUM(nw_order_details.UnitPrice*nw_order_details.Quantity) AS 'revenue'  FROM nw_suppliers
JOIN nw_products
ON nw_suppliers.SupplierID=nw_products.SupplierID
JOIN nw_order_details
ON nw_products.ProductID=nw_order_details.ProductID
GROUP BY nw_suppliers.SupplierID)
SELECT *, (revenue/SUM(revenue) OVER())*100 FROM t1 ORDER BY SupplierID;
"""

pd.read_sql_query(query, con=con)

C:\Users\HP\AppData\Local\Temp\ipykernel_73500\4099642386.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query(query, con=con)


,SupplierID,revenue,(revenue/SUM(revenue) OVER())*100
0,1,12504.80,2.119689
1,2,17816.05,3.019999
2,3,13464.00,2.282283
3,4,11360.20,1.925668
4,5,9355.80,1.585902
5,6,8598.80,1.457583
6,7,52198.70,8.848202
7,8,22743.50,3.855251
8,9,5073.60,0.860026
9,10,1830.60,0.310305


In [6]:
query = """WITH t1 AS (SELECT nw_suppliers.SupplierID, COUNT(*) AS 'count', SUM(COUNT(*)) OVER() AS 'total'  FROM nw_suppliers
JOIN nw_products
ON nw_suppliers.SupplierID=nw_products.SupplierID
JOIN nw_order_details
ON nw_products.ProductID=nw_order_details.ProductID
GROUP BY nw_suppliers.SupplierID)
SELECT SupplierID ,(count/total)*100 FROM t1;"""

pd.read_sql_query(query, con=con)

C:\Users\HP\AppData\Local\Temp\ipykernel_73500\1873357335.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query(query, con=con)


,SupplierID,(count/total)*100
0,1,3.6
1,2,3.6
2,3,1.5
3,4,2.2
4,5,2.4
5,6,3.3
6,7,7.0
7,8,6.3
8,9,1.5
9,10,2.5


In [10]:
query = """With t1 AS (SELECT nw_products.ProductID, ProductName, year(OrderDate) AS 'year', SUM(Quantity) AS 'quantity'  FROM nw_order_details
JOIN nw_products
ON nw_order_details.ProductID=nw_products.ProductID
JOIN nw_orders
ON nw_orders.OrderID=nw_order_details.OrderID
GROUP BY nw_products.ProductID, year(OrderDate))
SELECT *,
ifnull((quantity - LAG(quantity) OVER(PARTITION BY ProductID ORDER BY Year))/IFNULL(LAG(quantity) OVER(PARTITION BY ProductID ORDER BY Year), 0)*100, 'no_year') AS 'percentage'
FROM t1;"""

pd.read_sql_query(query, con=con)

C:\Users\HP\AppData\Local\Temp\ipykernel_73500\1775870171.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query(query, con=con)


,ProductID,ProductName,year,quantity,percentage
0,1,Chai,1996,125.0,no_year
1,1,Chai,1997,136.0,8.8000
2,2,Chang,1996,226.0,no_year
3,2,Chang,1997,217.0,-3.9823
4,3,Aniseed Syrup,1996,30.0,no_year
...,...,...,...,...,...
145,75,Rhnbru Klosterbier,1997,386.0,221.6667
146,76,Lakkalikri,1996,146.0,no_year
147,76,Lakkalikri,1997,302.0,106.8493
148,77,Original Frankfurter grne Soe,1996,63.0,no_year
